# 03 — Regresioni polinomial, Ridge dhe Lasso

## Konteksti dhe problemi shkencor

Kalibrimi i një sensori tregon si rritja e fleksibilitetit sjell mbipërshtatje dhe si regularizimi e stabilizon problemin.

### Objektivat e të nxënit

- të ndërtoni veçori polinomiale;
- të dalloni kompleksitetin nga lineariteti në parametra;
- të implementoni Ridge;
- të përdorni cross-validation për regularizimin.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Kalibrim sensori me përgjigje jolineare

In [ ]:
df=pd.read_csv(DATA/'sensor_calibration.csv'); x=df.temperature_C.to_numpy(); y=df.voltage_V.to_numpy(); plt.scatter(x,y,s=18); plt.show()

## 2. Matrica polinomiale nga parimet e para

In [ ]:
from src.regression.manual import polynomial_design
for d in [1,2,5]: print(d,polynomial_design(x[:3],d))

## 3. OLS për disa shkallë

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
xt,xv,yt,yv=train_test_split(x,y,test_size=.3,random_state=0)
rows=[]
for d in range(1,11):
    pt=PolynomialFeatures(d,include_bias=False); Xt=pt.fit_transform(xt[:,None]); Xv=pt.transform(xv[:,None]); m=LinearRegression().fit(Xt,yt); rows.append((d,mean_squared_error(yt,m.predict(Xt)),mean_squared_error(yv,m.predict(Xv))))
pd.DataFrame(rows,columns=['degree','train_MSE','valid_MSE'])

In [ ]:
rr=pd.DataFrame(rows,columns=['degree','train_MSE','valid_MSE']); plt.plot(rr.degree,rr.train_MSE,marker='o',label='train'); plt.plot(rr.degree,rr.valid_MSE,marker='o',label='valid'); plt.yscale('log'); plt.legend(); plt.show()

## 4. Ridge nga formula e mbyllur

In [ ]:
d=8; P=PolynomialFeatures(d,include_bias=False); X=P.fit_transform(x[:,None]); mu=X.mean(0); sd=X.std(0); Xs=(X-mu)/sd
A=np.c_[np.ones(len(Xs)),Xs]
def ridge_closed(alpha):
    I=np.eye(A.shape[1]); I[0,0]=0; return np.linalg.solve(A.T@A+alpha*I,A.T@y)
for a in [0,0.01,1,100]: print(a,np.linalg.norm(ridge_closed(a)[1:]))

## 5. Krahasim me Ridge dhe Lasso në pipeline

In [ ]:
from sklearn.linear_model import Ridge,Lasso
from sklearn.model_selection import cross_val_score
for model in [LinearRegression(),Ridge(alpha=1),Lasso(alpha=.001,max_iter=20000)]:
    p=make_pipeline(PolynomialFeatures(6,include_bias=False),StandardScaler(),model)
    s=-cross_val_score(p,x[:,None],y,cv=5,scoring='neg_mean_squared_error')
    print(model.__class__.__name__,s.mean(),s.std())

## 6. Rrugët e regularizimit

In [ ]:
alphas=np.logspace(-4,3,40); norms=[]
Xp=PolynomialFeatures(8,include_bias=False).fit_transform(x[:,None]); Xp=StandardScaler().fit_transform(Xp)
for a in alphas: norms.append(np.linalg.norm(Ridge(alpha=a).fit(Xp,y).coef_))
plt.semilogx(alphas,norms); plt.xlabel('alpha'); plt.ylabel('||coef||'); plt.show()

## 7. Ndërveprime fizike

Jo çdo term polinomial është i arsyeshëm. Në probleme me disa madhësi, produkte si $VT$, $B^2$ ose $xv$ mund të kenë motivim fizik. Ndërtimi i veçorive duhet të udhëhiqet nga mekanizmi, jo vetëm nga score.

## 8. Rast dështimi: ekstrapolimi

In [ ]:
train=df[df.temperature_C<55]; test=np.linspace(55,120,120);
for d in [2,8]:
 p=make_pipeline(PolynomialFeatures(d,include_bias=False),LinearRegression()).fit(train[['temperature_C']],train.voltage_V); plt.plot(test,p.predict(test[:,None]),label=f'd={d}')
plt.legend(); plt.title('Ekstrapolimi jashtë diapazonit'); plt.show()

## 9. Ushtrime

1. Derivoni Ridge nga funksioni i humbjes.
2. Krahasoni shkallët 1–12 me nested CV.
3. Ndryshoni standardizimin dhe vëzhgoni Lasso.
4. Ndërtoni një problem me dy veçori të korreluara dhe studioni stabilitetin e Lasso.

## 10. Shembull i plotë i punuar

Zgjidhni shkallën dhe alpha vetëm me të dhënat e zhvillimit; hapni testin final në fund. Raportoni kurbën e validimit dhe justifikoni zgjedhjen.

## 11. Playground

Ndryshoni zhurmën, shkallën reale të polinomit dhe numrin e pikave. Kur fillon overfitting? A ndryshon përgjigjja me madhësinë e mostrës?